# Project 1 — Step 1: the simple logit model

**Advanced Industrial Organization**

This notebook covers the first four steps of the empirical part: loading the data,
defining the market, estimating a simple logit demand model, and working out what
the estimates imply for elasticities and markups.

Requires `pandas`, `numpy`, `statsmodels`. Run the cells in order.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 1. Load the data

`cars.csv` is the **balanced** panel: J = 40 cars in every one of the
N = 5 x 30 = 150 country-year markets, so 6,000 rows.
(`cars.xlsx` is the unbalanced version, where J varies — we do not use it,
because the counterfactual code later assumes a fixed J.)

In [ ]:
cars = pd.read_csv("cars.csv")

MARKET = {1: "Belgium", 2: "France", 3: "Germany", 4: "Italy", 5: "UK"}
SEGMENT = {1: "subcompact", 2: "compact", 3: "intermediate",
           4: "standard", 5: "luxury"}

cars["country"] = cars["ma"].map(MARKET)
cars["segment"] = cars["cla"].map(SEGMENT)
cars["mkt"] = cars["ma"].astype(str) + "_" + cars["ye"].astype(str)

print(cars.shape)
print(f"{cars.mkt.nunique()} markets, {cars.groupby('mkt').size().unique()} cars each")
cars[["ye", "ma", "country", "brand", "model", "segment", "qu", "pr", "princ",
      "we", "li", "hp", "home", "frm"]].head()

## 2. Market size and market shares

**The choice to argue for in the report.** The data give `qu` (units sold) and
`pop` (population), but not the market *size* — that is our modelling choice.
Here we treat the buying unit as a **household** and set

$$M_t = \text{pop}_t / 3$$

since average household size in these five countries is roughly 2.5–3.3 over the
sample. The outside good is "no new car this year".

Change `HH_SIZE` to see how sensitive the results are — we check that formally
at the end of the notebook.

In [ ]:
HH_SIZE = 3.0                              # <-- YOUR CHOICE

cars["market_size"] = cars["pop"] / HH_SIZE
cars["s"] = cars["qu"] / cars["market_size"]
cars["s0"] = 1.0 - cars.groupby("mkt")["s"].transform("sum")

print(f"Outside share: {cars.s0.min():.3f} to {cars.s0.max():.3f} "
      f"(mean {cars.s0.mean():.3f})")
print(f"Inside share:  {1 - cars.s0.mean():.3f}  "
      f"-> a household buys one of these 40 cars every "
      f"{1 / (1 - cars.s0.mean()):.1f} years")

assert (cars.s > 0).all(), "some market shares are zero or negative"
assert (cars.s0 > 0).all(), "market size too small: inside shares sum above 1"

In [ ]:
# shares by country, as a sanity check
cars.groupby("country")["s"].describe()[["mean", "min", "max"]].round(4)

## 3. Dependent variable and regressors

Berry's (1994) inversion turns the logit model into a linear regression:

$$\ln s_{jt} - \ln s_{0t} = x_{jt}'\beta - \alpha p_{jt} + \xi_{jt}$$

**Price.** We use `princ` = price / GDP per capita, in levels. This makes prices
comparable across five currencies, thirty years of inflation and different income
levels, and it makes utility quasi-linear in the income share the car costs.
`logp` is kept for the robustness check.

**Characteristics.** Weight, fuel consumption and the `home` dummy are the core;
power, footprint and displacement are added as further controls. Note that `ac`
(acceleration) is **not** used: it is missing for 986 observations, and dropping
those rows would break the balanced panel.

In [ ]:
cars["y"] = np.log(cars["s"]) - np.log(cars["s0"])     # Berry inversion
cars["p"] = cars["princ"]                              # price / income p.c.
cars["logp"] = np.log(cars["p"])

cars["we_t"] = cars["we"] / 1000.0                     # weight, tonnes
cars["hp_10"] = cars["hp"] / 10.0                      # power, 10 kW
cars["size"] = cars["le"] * cars["wi"] / 10000.0       # footprint, m2
cars["cy_l"] = cars["cy"] / 1000.0                     # displacement, litres

# check for missing values before regressing - a NaN here fails silently later
use = ["y", "p", "logp", "home", "we_t", "li", "hp_10", "size", "cy_l"]
print(cars[use].isna().sum().to_string())
print()
print(cars[["p", "home", "we_t", "li", "hp_10", "size", "cy_l"]].describe().round(3).to_string())

## 4. Estimation

Brand fixed effects absorb the permanent quality of a marque; country and year
fixed effects absorb market size, macro conditions and common trends. Standard
errors are clustered by market (country-year), since $\xi_{jt}$ is plausibly
correlated across the cars sold in the same market.

In [ ]:
X = "home + we_t + li + hp_10 + size + cy_l"
FE = "C(brand) + C(ma) + C(ye)"

def run(price_var, data=None):
    d = cars if data is None else data
    formula = f"y ~ {price_var} + {X} + {FE}"
    return smf.ols(formula, data=d).fit(cov_type="cluster",
                                        cov_kwds={"groups": d["mkt"]})

logit = run("p")
alpha = -logit.params["p"]

show = ["p", "home", "we_t", "li", "hp_10", "size", "cy_l"]
table = pd.DataFrame({"coef": logit.params[show],
                      "se": logit.bse[show],
                      "t": logit.tvalues[show],
                      "p-value": logit.pvalues[show]})
print(table.round(3).to_string())
print(f"\nalpha = {alpha:.3f}   R2 = {logit.rsquared:.3f}   N = {int(logit.nobs)}")

### Reading the coefficients

- **`home`** is the direct measure of home bias: a domestic car has a mean utility
  about 1.2 higher than an otherwise identical import.
- **`li`** is fuel *consumption*, so a negative coefficient is the expected sign —
  but it is not significant here, most likely because it is strongly collinear with
  weight and engine size.
- **`hp_10`** comes out negative. That is not a sign error: conditional on price and
  weight, extra horsepower is not something buyers reward.

## 5. Elasticities

$$\eta_{jj} = -\alpha p_j (1 - s_j), \qquad \eta_{jk} = \alpha p_k s_k$$

Note what the cross-elasticity does **not** contain: the index $j$. The substitution
from a Fiat Punto to a VW Golf is identical to the substitution from a Mercedes
S-class to a VW Golf. This is the IIA property, and it is the reason we extend the
model in the next notebook.

In [ ]:
cars["own_el"] = -alpha * cars["p"] * (1 - cars["s"])
cars["cross_el"] = alpha * cars["p"] * cars["s"]     # same for every rival j

print("Own-price elasticities")
print(cars["own_el"].describe(percentiles=[.1, .5, .9]).round(3).to_string())
print(f"\nMean cross-price elasticity: {cars.cross_el.mean():.5f}")
print()
print("By segment:")
print(cars.groupby("segment")["own_el"].agg(["mean", "min", "max"]).round(3).to_string())

In [ ]:
# the full elasticity matrix for one market, to show in the report
def elasticity_matrix(mkt="4_99", top=6):
    sub = cars[cars.mkt == mkt].nlargest(top, "s")
    names = (sub["brand"].str.title() + " " + sub["model"].str.title()).values
    s, p = sub["s"].values, sub["p"].values
    E = alpha * np.tile(p * s, (len(s), 1))          # cross elasticities
    np.fill_diagonal(E, -alpha * p * (1 - s))        # own elasticities
    return pd.DataFrame(E, index=names, columns=names)

print("Italy 1999, six best-selling cars - simple logit")
elasticity_matrix().round(4)

Every off-diagonal entry in a given **column** is the same number. That is IIA
made visible, and it is worth putting this matrix in the output appendix next to
the nested logit version.

## 6. The markup diagnostic

For a single-product firm the Bertrand first-order condition gives

$$p_j - mc_j = \frac{1}{\alpha (1 - s_j)}$$

If demand is inelastic ($|\eta_{jj}| < 1$), this markup exceeds the price and the
implied marginal cost is **negative** — which no firm can have. This is the key
diagnostic for the simple logit, and the argument for extending the model.

In [ ]:
cars["markup"] = 1.0 / (alpha * (1 - cars["s"]))
cars["mc"] = cars["p"] - cars["markup"]
cars["lerner"] = cars["markup"] / cars["p"]

print(f"Median Lerner index:        {cars.lerner.median():.3f}")
print(f"Share of cars with mc < 0:  {(cars.mc < 0).mean():.1%}")
print(f"Share with |elasticity| < 1:{(cars.own_el > -1).mean():.1%}")
print()
print("Share of cars with mc < 0, by segment:")
print(cars.groupby("segment")["mc"].apply(lambda x: (x < 0).mean()).round(3).to_string())

## 7. Robustness

Two of the practical choices deserve a sentence each in the report.

In [ ]:
print("Market size")
for hh in [2.0, 3.0, 4.0]:
    d = cars.copy()
    d["s"] = d["qu"] / (d["pop"] / hh)
    d["s0"] = 1.0 - d.groupby("mkt")["s"].transform("sum")
    d["y"] = np.log(d["s"]) - np.log(d["s0"])
    m = run("p", data=d)
    print(f"  pop/{hh:.0f}:  alpha = {-m.params['p']:.3f}  "
          f"(se {m.bse['p']:.3f}),  outside share {d.s0.mean():.3f}")

In [ ]:
print("Price in logs instead of levels")
m = run("logp")
a_log = -m.params["logp"]
el_log = -a_log * (1 - cars["s"])
print(f"  alpha = {a_log:.3f} (se {m.bse['logp']:.3f})")
print(f"  mean own-price elasticity = {el_log.mean():.3f}")
print(f"  share with |elasticity| < 1: {(el_log > -1).mean():.1%}")

In [ ]:
print("Dropping the fixed effects one dimension at a time")
for fe in ["C(brand) + C(ma) + C(ye)", "C(ma) + C(ye)", "C(brand) + C(mkt)", "1"]:
    m = smf.ols(f"y ~ p + {X} + {fe}", data=cars).fit()
    print(f"  FE = {fe:28s} alpha = {-m.params['p']:.3f}  R2 = {m.rsquared:.3f}")

## 8. What goes into the report

| Result | Where it goes |
| --- | --- |
| Coefficient table | Output appendix, Table 1, column 1 |
| Elasticity summary | Output appendix, Table 2 |
| Elasticity matrix, one market | Output appendix, Table 3 |
| `alpha` stable across market sizes | Section 2, one sentence |
| Share of cars with `mc < 0` | Section 4, the argument for extending the model |

**Next step:** the nested logit, which relaxes IIA by letting cars within a segment
be closer substitutes.